In [19]:
# First, we install the official IBL ONE (Open Neurophysiology Environment) API
# and basic diagnostic packages.
!pip install ONE-api ibllib pandas numpy scikit-learn matplotlib seaborn tqdm --quiet

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from brainbox.io.one import SessionLoader

print("Environment setup and libraries loaded successfully.")

Environment setup and libraries loaded successfully.


In [20]:
from one.api import ONE
ONE.setup(base_url='https://openalyx.internationalbrainlab.org', silent=True)
one = ONE(password='international')

Connected to https://openalyx.internationalbrainlab.org as user "intbrainlab"


In [21]:
# ==============================================================================
# STEP 2.7: BUILD & RENDER THE SESSION DIRECTORY DASHBOARD
# ==============================================================================
import pandas as pd
from IPython.display import display, HTML

print("Querying Alyx database for matching sessions...")

# 1. Query the database using our verified filters
all_matching_eids = one.search(
    projects='brainwide',                 # Standardized expert BWM sessions
    datasets=[
        'probes.description.json',        # Neural spike data
        '_ibl_leftCamera.features.pqt'     # Pupil tracking features
    ],
    lab=['churchlandlab', 'mainenlab', 'cortexlab', 'angelakilab'] # 3D Visualizer synced labs
)

# Robust Fallback: If no sessions in chosen labs, search across all labs
if len(all_matching_eids) == 0:
    print("⚠️ No sessions found in target labs. Broadening search to all Brain-Wide Map labs...")
    all_matching_eids = one.search(
        projects='brainwide',
        datasets=[
            'probes.description.json',
            '_ibl_leftCamera.features.pqt'
        ]
    )

matching_eid_strings = [str(eid) for eid in all_matching_eids]

# 2. Safely compile metadata if we have entries
session_metadata_list = []
if len(matching_eid_strings) > 0:
    print(f"Acquired {len(matching_eid_strings)} EIDs. Fetching subject details...")
    for eid_str in matching_eid_strings:
        try:
            details = one.get_details(eid_str)
            session_metadata_list.append({
                'Subject (Mouse)': details['subject'],
                'Date': details['start_time'][:10],
                'Lab': details['lab'],
                'EID': eid_str
            })
        except Exception:
            session_metadata_list.append({
                'Subject (Mouse)': 'Unknown',
                'Date': 'Unknown',
                'Lab': 'Unknown',
                'EID': eid_str
            })

    # Create master DataFrame
    df_sessions = pd.DataFrame(session_metadata_list)

    # Save backup locally
    csv_filename = "expert_mice_sessions.csv"
    df_sessions.to_csv(csv_filename, index=False)
    print(f"💾 Registry successfully backed up to offline spreadsheet: '{csv_filename}'\n")

    total_sessions = len(df_sessions)
    unique_mice = df_sessions['Subject (Mouse)'].nunique()
    unique_labs = df_sessions['Lab'].nunique()
else:
    # Empty State Configuration to avoid Pandas styling KeyError crash
    df_sessions = pd.DataFrame(columns=['Subject (Mouse)', 'Date', 'Lab', 'EID'])
    total_sessions, unique_mice, unique_labs = 0, 0, 0
    print("❌ Active database search returned zero matching records.\n")





Querying Alyx database for matching sessions...
⚠️ No sessions found in target labs. Broadening search to all Brain-Wide Map labs...
Acquired 458 EIDs. Fetching subject details...
💾 Registry successfully backed up to offline spreadsheet: 'expert_mice_sessions.csv'



In [22]:
df_sessions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 458 entries, 0 to 457
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Subject (Mouse)  458 non-null    object
 1   Date             458 non-null    object
 2   Lab              458 non-null    object
 3   EID              458 non-null    object
dtypes: object(4)
memory usage: 14.4+ KB


In [23]:
print(df_sessions)

    Subject (Mouse)        Date                 Lab  \
0            MFD_09  2023-10-19  churchlandlab_ucla   
1            MFD_08  2023-09-08  churchlandlab_ucla   
2            MFD_08  2023-09-07  churchlandlab_ucla   
3           NR_0029  2023-09-07        steinmetzlab   
4           NR_0029  2023-09-05        steinmetzlab   
..              ...         ...                 ...   
453           KS016  2019-12-05           cortexlab   
454           KS014  2019-12-05           cortexlab   
455           KS014  2019-12-03           cortexlab   
456   ibl_witten_13  2019-11-27           wittenlab   
457   ibl_witten_13  2019-11-26           wittenlab   

                                      EID  
0    ebce500b-c530-47de-8cb1-963c552703ea  
1    a7eba2cf-427f-4df9-879b-e53e962eae18  
2    5ae68c54-2897-4d3a-8120-426150704385  
3    3a3ea015-b5f4-4e8b-b189-9364d1fc7435  
4    d85c454e-8737-4cba-b6ad-b2339429d99b  
..                                    ...  
453  6cf2a88a-515b-4f7f-89a2-7d

Run next cell to download csv file ⬇️

In [24]:
from google.colab import files

# Triggers a browser prompt to download your saved registry file
files.download("expert_mice_sessions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>